# Phase 4: Court Judgment Prediction: Encoder-decoder (6x6)

This is the **Court Judgment Prediction** part of the full `encdec` notebook (`notebooks/CS781_encdec_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Court Judgement Prediction

In [ ]:
import pickle, os
P4_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_4"
C = pickle.load(open(f"{P4_DIR}/cjpe_tok_cache.pkl", "rb"))
print({k: len(v["ids"]) for k, v in C.items()}, "✅ Phase-4 data ready")

{'train': 37387, 'dev': 3505, 'test': 1517} ✅ Phase-4 data ready


In [7]:
import json, time, math, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

CH = 1024
MODES = {"first1": 1, "last1": 1, "first4": 4, "last4": 4, "ht4": 4}   # value = max chunks per doc

# ---------- document -> chunks (from the cached head/tail tokens) ----------
def get_chunks(split, i, mode):
    d = C[split]; n = int(d["n"][i]); head, tail = d["head"][i], d["tail"][i]
    if   mode == "first1": x = head[:CH]
    elif mode == "last1":  x = tail[-CH:]
    elif mode == "first4": x = head[:4*CH]
    elif mode == "last4":  x = tail[-4*CH:]
    elif mode == "ht4":    x = head if n <= 4*CH else np.concatenate([head[:2*CH], tail[-2*CH:]])
    L = len(x)
    if mode.startswith("last"):                      # align chunks to the END of the judgment
        ends = list(range(L, 0, -CH))[::-1]
        return [x[max(0, e-CH):e] for e in ends]
    return [x[s:s+CH] for s in range(0, L, CH)]

def collate_cj(split, idxs, mode):
    seqs, owner = [], []
    for j, i in enumerate(idxs):
        cs = get_chunks(split, i, mode)
        if not cs or len(cs[0]) == 0: cs = [np.zeros(1, np.uint16)]
        for c in cs: seqs.append(c); owner.append(j)
    T = max(len(s) for s in seqs)
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.from_numpy(s.astype(np.int64))
    lens = torch.tensor([len(s) for s in seqs])
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(idxs)

# ---------- model: Phase-1 LM -> chunk features -> mean over chunks -> 2-class head ----------
class CJModel(RRModel):
    def forward(self, ids, lens, owner, n_docs):
        H = self.hidden(ids)                                            # (chunks, T, d)
        B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        mean = (H.float() * mask).sum(1) / lens[:, None]
        last = H[torch.arange(B, device=H.device), lens - 1].float()
        f = torch.cat([last, mean], -1)                                 # (chunks, 2d)
        D = torch.zeros(n_docs, f.size(1), device=f.device).index_add_(0, owner, f)
        D = D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]
        return self.head(self.drop(D))

def cj_from(rr):
    """Turn a Phase-3 RRModel (13-label head) into a CJPE model: same LM, fresh 2-class head."""
    d = rr.head.in_features // 2
    return CJModel(rr.lm, d, 2, 0.1).to(device)

# ---------- inference / metrics ----------
@torch.no_grad()
def cj_logits(model, split, mode, idxs=None, chunk_batch=32):
    model.eval()
    idxs = np.arange(len(C[split]["ids"])) if idxs is None else np.asarray(idxs)
    bs = max(1, chunk_batch // MODES[mode]); out = np.zeros((len(idxs), 2), np.float32)
    for s in range(0, len(idxs), bs):
        b = idxs[s:s+bs]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lg = model(*collate_cj(split, b, mode))
        out[s:s+len(b)] = lg.float().cpu().numpy()
    return out

def cj_scores(y, lg, thr=0.5):
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pr = (p1 > thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pr, average="macro", zero_division=0)
    return dict(f1=Fm, p=P, r=R, cm=confusion_matrix(y, pr, labels=[0, 1]))

def best_thr(y, lg):
    best = (0.5, -1.0)
    for t in np.arange(0.2, 0.81, 0.02):
        f = cj_scores(y, lg, t)["f1"]
        if f > best[1]: best = (float(t), f)
    return best

# ---------- training (CE, or KD from cached teacher logits) ----------
def train_cj(model, name, mode, epochs=2, lr=5e-5, head_lr=1e-3, wd=0.01, bs_chunks=16,
             n_train=None, kd=None, alpha=0.5, T=2.0, evals_per_epoch=2, patience=3,
             class_weight=True, debug=False, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    ytr = C["train"]["y"]; ydv = C["dev"]["y"]
    dev_idx = np.arange(200) if debug else np.arange(len(ydv))
    tr_idx = rng.permutation(len(ytr))[: (128 if debug else (n_train or len(ytr)))]
    if debug: epochs, evals_per_epoch = 1, 1
    cw = None
    if class_weight:
        cnt = np.bincount(ytr, minlength=2); cw = torch.tensor(len(ytr) / (2 * cnt), dtype=torch.float, device=device)
    kd_t = torch.tensor(kd, dtype=torch.float) if kd is not None else None

    head_p = [p for p in model.head.parameters()]
    hid = {id(p) for p in head_p}
    body_p = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body_p, "lr": lr}, {"params": head_p, "lr": head_lr}], weight_decay=wd)
    bs = max(1, bs_chunks // MODES[mode]); spe = math.ceil(len(tr_idx) / bs); total = spe * epochs
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    every = max(1, spe // evals_per_epoch)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{name}] mode={mode} train_docs={len(tr_idx)} steps={total} trainable={n_tr/1e6:.2f}M / {n_all/1e6:.1f}M")

    path = f"{P4_DIR}/cj_{name}_best.pt"; best = (-1.0, None); bad = 0; step = 0; t0 = time.time()
    torch.cuda.reset_peak_memory_stats(); hist = []; done = False
    for ep in range(epochs):
        perm = rng.permutation(tr_idx); model.train()
        for s in range(0, len(perm), bs):
            b = perm[s:s+bs]
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(*collate_cj("train", b, mode)).float()
            y = torch.tensor(ytr[b], device=device)
            loss = F.cross_entropy(lg, y, weight=cw)
            if kd_t is not None:
                t_lg = kd_t[b].to(device)
                kl = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(t_lg / T, -1), reduction="batchmean") * T * T
                loss = (1 - alpha) * loss + alpha * kl
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); step += 1
            if step == 20: print(f"  ~{(time.time()-t0)/20*total/60:.0f} min training total (excl. dev evals)")
            if step % 100 == 0: print(f"  step {step}/{total} loss {loss.item():.4f}")
            if step % every == 0 or step == total:
                lgd = cj_logits(model, "dev", mode, dev_idx); sc = cj_scores(ydv[dev_idx], lgd)
                hist.append((step, sc["f1"])); model.train()
                print(f"  eval@{step}: macro-F1 {sc['f1']:.4f}  P {sc['p']:.4f}  R {sc['r']:.4f}  ({(time.time()-t0)/60:.1f} min)")
                if sc["f1"] > best[0]:
                    best = (sc["f1"], step); bad = 0
                    torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, path)
                else:
                    bad += 1
                    if bad >= patience: done = True; break
        if done: break
    if os.path.exists(path):
        model.load_state_dict(torch.load(path), strict=False)
    lgd = cj_logits(model, "dev", mode, dev_idx); y = ydv[dev_idx]
    sc = cj_scores(y, lgd); thr, f_thr = best_thr(y, lgd)
    res = dict(name=name, mode=mode, f1=sc["f1"], p=sc["p"], r=sc["r"], cm=sc["cm"].tolist(), thr=thr, f1_thr=f_thr,
               minutes=(time.time()-t0)/60, peak_gb=torch.cuda.max_memory_allocated()/1e9,
               trainable=n_tr, total=n_all, best_step=best[1], dev_logits=lgd)
    print(f"✅ [{name}] dev macro-F1 {sc['f1']:.4f} (P {sc['p']:.4f} R {sc['r']:.4f}) | tuned thr {thr:.2f} → {f_thr:.4f} "
          f"| {res['minutes']:.1f} min, peak {res['peak_gb']:.1f} GB\n   confusion [[TN,FP],[FN,TP]] = {res['cm']}")
    return res

# ---------- outputs ----------
def write_pred(model, name, mode, thr=0.5):
    lg = cj_logits(model, "test", mode)
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pred = (p1 > thr).astype(int)
    out = [{"id": i, "pred": int(p)} for i, p in zip(C["test"]["ids"], pred)]
    assert len(out) == len(C["test"]["ids"]) == 1517
    f = f"{P4_DIR}/prediction_{name}.json"; json.dump(out, open(f, "w"))
    np.save(f"{P4_DIR}/test_logits_{name}.npy", lg)
    print(f"wrote {f} | preds 0/1 = {np.bincount(pred, minlength=2).tolist()}  (rename to prediction.json when uploading)")

def save_kd_logits(model, name, mode):
    lg = cj_logits(model, "train", mode); np.save(f"{P4_DIR}/kd_logits_{name}.npy", lg); return lg

print("✅ Phase-4 toolkit ready")

✅ Phase-4 toolkit ready


In [ ]:
ENCDEC_CKPT = f"{DRIVE_PROJECT_DIR}/encdec_6x6_checkpoints/best.pt"
assert os.path.exists(ENCDEC_CKPT), ENCDEC_CKPT
for n in ["EncoderDecoderLM", "RRModel", "train_cj", "collate_cj", "write_pred", "C"]:
    assert n in globals(), f"missing {n}: run the earlier cells first"

class CJEncModel(RRModel):
    """CJPE on the enc-dec ENCODER. Each chunk runs alone (no attention mask), features = [last ; mean], mean over chunks."""
    def forward(self, ids, lens, owner, n_docs):
        feats = []
        for j in range(ids.shape[0]):
            n = int(lens[j]); H = self.lm.encode(ids[j:j + 1, :n])[0].float()       # (n, d)
            feats.append(torch.cat([H[n - 1], H.mean(0)], -1))
        f = torch.stack(feats)                                                        # (chunks, 2d)
        D = torch.zeros(n_docs, f.size(1), device=f.device).index_add_(0, owner, f)
        D = D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]
        return self.head(self.drop(D))

def build_encdec_cj(dropout=0.1):
    lm_ = EncoderDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=640, n_enc_layer=6, n_dec_layer=6,
                           n_head=10, n_kv_head=5, max_src_len=1024, max_tgt_len=320,
                           bos_id=tokenizer.token_to_id("<|endoftext|>"), tie_weights=True)
    ck = torch.load(ENCDEC_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)              # pretrained Phase-1 weights
    m = CJEncModel(lm_, 640, 2, dropout).to(device)
    m.head = nn.Linear(2 * 640, 2).to(device)                               # fresh 2-class head
    nn.init.normal_(m.head.weight, std=0.02); nn.init.zeros_(m.head.bias)
    return m

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

def inject_lora_enc(rr, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in rr.parameters(): p.requires_grad_(False)
    for layer in rr.lm.enc_layers:                                           # encoder self-attention only
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, LoRALinear(base, r, alpha).to(base.weight.device))
    for p in rr.head.parameters(): p.requires_grad_(True)

DEBUG   = False         # True first (~1-2 min), then False
MODE    = "last4"
N_TRAIN = None          # None = all 37k docs; e.g. 20000 to roughly halve the time
CJ = globals().get("CJ", {})

random.seed(1); torch.manual_seed(1)
m = build_encdec_cj()
batch = collate_cj("dev", [0, 1], "last1"); m.eval()
with torch.no_grad(): before = m(*batch)
inject_lora_enc(m, 8, 16)
with torch.no_grad(): d = (before - m(*batch)).abs().max().item()
assert d < 1e-3, f"LoRA is not identity at init ({d})"; print(f"[lora] identity at init OK ({d:.1e})")

NAME = f"encdec_lora_{MODE}"
CJ[NAME] = train_cj(m, NAME, MODE, epochs=2, lr=2e-3, head_lr=1e-3, n_train=N_TRAIN, debug=DEBUG)
if not DEBUG: write_pred(m, NAME, MODE)

[lora] identity at init OK (0.0e+00)
[encdec_lora_last4] mode=last4 train_docs=37387 steps=18694 trainable=0.22M / 84.1M
  ~124 min training total (excl. dev evals)
  step 100/18694 loss 0.7445
  step 200/18694 loss 0.7132
  step 300/18694 loss 0.7003
  step 400/18694 loss 0.5105
  step 500/18694 loss 0.9901
  step 600/18694 loss 0.6900
  step 700/18694 loss 0.6303
  step 800/18694 loss 0.6798
  step 900/18694 loss 0.6215
  step 1000/18694 loss 0.7401
  step 1100/18694 loss 0.7784
  step 1200/18694 loss 0.7222
  step 1300/18694 loss 0.7953
  step 1400/18694 loss 0.7029
  step 1500/18694 loss 0.6953
  step 1600/18694 loss 0.6390
  step 1700/18694 loss 0.7365
  step 1800/18694 loss 0.5708
  step 1900/18694 loss 0.7325
  step 2000/18694 loss 0.6880
  step 2100/18694 loss 0.5559
  step 2200/18694 loss 0.6576
  step 2300/18694 loss 1.4834
  step 2400/18694 loss 0.8432
  step 2500/18694 loss 0.5866
  step 2600/18694 loss 0.7830
  step 2700/18694 loss 2.6395
  step 2800/18694 loss 1.0313
  st